## Name: Mandar Sanju Bavdane

### CWID: 20041513                  Subject: FE_520 -- Assignment 1

In [39]:
import re
from collections import Counter


def fix_ticker(symbol):
    symbol = symbol.strip().upper()
    symbol = symbol.replace("–", "-").replace("—", "-")
    if symbol == "BRK-B":
        return "BRK.B"
    return symbol


## Question 1

Produce tickers: uppercase tickers from raw, where all [,;|] are treated as commas, whitespace is trimmed, blanks removed, and BRK-B is canonicalized to BRK.B (case-insensitive). No loops. re allowed.

In [ ]:
# Input
raw = " aapl ,msft, GOOG ;BRK.B| tsla , spy ,aapl, Ibm ; nvda, brk-b, ^GSPC "
universe = ["AAPL", "MSFT", "GOOGL", "GOOG", "BRK.B", "TSLA", "SPY", "IBM", "NVDA", "^GSPC", "META", "AMZN"]

parts = re.split(r"[,;|]", raw)
tickers = list(map(fix_ticker, filter(lambda s: s.strip(), parts)))

unique_ordered = list(dict.fromkeys(tickers))
valid_universe = list(filter(lambda s: s in universe, unique_ordered))
# difference of set identifies unrecognized symbols and sorted() puts them in order
dropped = sorted(set(unique_ordered) - set(universe)) 

# sorted() puts ^GSPC last; the sample wants it first
valid_sorted = sorted(valid_universe, key=lambda s: (not s.startswith("^"), s))
# Join each result into the requested status format
status = "VALID:" + ",".join(valid_sorted) + "; DROPPED:" + ",".join(dropped)

print("tickers =", tickers)
print("unique_ordered =", unique_ordered)
print("valid_universe =", valid_universe)
print("dropped =", dropped)
print("status =", status)
print()


tickers = ['AAPL', 'MSFT', 'GOOG', 'BRK.B', 'TSLA', 'SPY', 'AAPL', 'IBM', 'NVDA', 'BRK.B', '^GSPC']
unique_ordered = ['AAPL', 'MSFT', 'GOOG', 'BRK.B', 'TSLA', 'SPY', 'IBM', 'NVDA', '^GSPC']
valid_universe = ['AAPL', 'MSFT', 'GOOG', 'BRK.B', 'TSLA', 'SPY', 'IBM', 'NVDA', '^GSPC']
dropped = []
status = VALID:^GSPC,AAPL,BRK.B,GOOG,IBM,MSFT,NVDA,SPY,TSLA; DROPPED:



## Question 2 — Parse weights and classify notes (20 points)

Produce unique ordered: first occurrence of each symbol from tickers, order preserved. No loop

In [45]:
targets = """
AAPL: 25%, MSFT:25 % , nvda :  30% ,   GOOG:10%,   brk-b:10%
"""
notes = [
    "AAPL split-adjusted; BUYbacks ongoing",
    "MSFT AI capex: strong; HOLD raised to BUY",
    "NVDA valuation-debated; DOWNGRADE risk",
    "GOOG ads cyclical-HOLD",
    "BRK.B cash pile record; BUY",
]

found = re.findall(r"([A-Za-z][A-Za-z.\-]*)\s*:\s*([0-9.]+)\s*%", targets.replace("–", "-"))
pairs = sorted(map(lambda m: (fix_ticker(m[0]), float(m[1])), found))

# Sum the numeric weights
total = sum(map(lambda p: p[1], pairs))
if abs(total - 100) <= 0.01:
    weights_status = "WEIGHTS OK (100.00)"
else:
    weights_status = f"WEIGHTS ERROR (sum={total:.2f})"


def names_with(pattern):
    hits = filter(lambda note: re.search(pattern, note, re.I), notes)
    return sorted(map(lambda note: fix_ticker(note.split()[0]), hits))

# Return unique ticker lists in alphabetical order without any explicit loops
buy_list = names_with(r"\bBUY(?:BACKS)?\b")
hold_list = names_with(r"\bHOLD\b")
risk_flags = names_with(r"\bDOWNGRADE\b")

coverage = ",".join(map(lambda p: p[0], pairs))
memo = (
    f"Coverage: {coverage}\n"
    f"BUY: {','.join(buy_list)}\n"
    f"HOLD: {','.join(hold_list)} | RISK: {','.join(risk_flags)}"
)

buy_set = set(buy_list)
hold_set = set(hold_list)
risk_set = set(risk_flags)


def one_row(pair):
    name, wt = pair
    buy = "Y" if name in buy_set else ""
    hold = "Y" if name in hold_set else ""
    risk = "Y" if name in risk_set else ""
    return f'"{name}","{wt:.2f}","{buy}","{hold}","{risk}"'


csv_header = '"Ticker","Target(%)","BUY","HOLD","RISK"'
csv_row = "\n".join(map(one_row, pairs))

print("pairs =", pairs)
print("weights_status =", weights_status)
print("buy_list =", buy_list)
print("hold_list =", hold_list)
print("risk_flags =", risk_flags)
print("\n")
print("Memo\n" + memo)
print(csv_header)
print(csv_row)
print()


pairs = [('AAPL', 25.0), ('BRK.B', 10.0), ('GOOG', 10.0), ('MSFT', 25.0), ('NVDA', 30.0)]
weights_status = WEIGHTS OK (100.00)
buy_list = ['AAPL', 'BRK.B', 'MSFT']
hold_list = ['GOOG', 'MSFT']
risk_flags = ['NVDA']


Memo
Coverage: AAPL,BRK.B,GOOG,MSFT,NVDA
BUY: AAPL,BRK.B,MSFT
HOLD: GOOG,MSFT | RISK: NVDA
"Ticker","Target(%)","BUY","HOLD","RISK"
"AAPL","25.00","Y","",""
"BRK.B","10.00","Y","",""
"GOOG","10.00","","Y",""
"MSFT","25.00","Y","Y",""
"NVDA","30.00","","","Y"



## Question 3

Produce valid_universe: keep only symbols that exist in universe, order as in unique_ordered.  

In [ ]:
sec = """
We returned $29.5 billion to shareholders via buybacks and dividends in FY2024.
Operating margin expanded by 120 bps to 31.2%. Cash and marketable securities totaled $61.8 billion.
We reduced diluted shares outstanding by ~1.8%. Long-term debt declined to $105.3 billion.
"""
# Capture the numeric part only when it is followed by the word "billion"
dollar_billions = sorted(set(re.findall(r"\$(\d+\.\d+) billion", sec)))

percents = list(dict.fromkeys(re.findall(r"\d+(?:\.\d+)?%", sec)))
metrics_line = f"BILLIONS:[{','.join(dollar_billions)}] | PCTS:[{','.join(percents)}]"

sec_approx = re.sub(r"~(?=\d+(?:\.\d+)?%)", "≈", sec)
sec_clean = re.sub(r"\s+", " ", sec_approx).strip() + "\n"

print("dollar_billions =", dollar_billions)
print("percents =", percents)
print("metrics_line =", metrics_line)
print("sec_approx =", sec_approx)
print("sec_clean =", sec_clean)


dollar_billions = ['105.3', '29.5', '61.8']
percents = ['31.2%', '1.8%']
metrics_line = BILLIONS:[105.3,29.5,61.8] | PCTS:[31.2%,1.8%]
sec_approx = 
We returned $29.5 billion to shareholders via buybacks and dividends in FY2024.
Operating margin expanded by 120 bps to 31.2%. Cash and marketable securities totaled $61.8 billion.
We reduced diluted shares outstanding by ≈1.8%. Long-term debt declined to $105.3 billion.

sec_clean = We returned $29.5 billion to shareholders via buybacks and dividends in FY2024. Operating margin expanded by 120 bps to 31.2%. Cash and marketable securities totaled $61.8 billion. We reduced diluted shares outstanding by ≈1.8%. Long-term debt declined to $105.3 billion.



## Question 4

Produce dropped: the set of symbols in unique_ordered not found in universe as a sorted list (may be 
empty).

In [43]:
csv_text = """Date,Close
2025-09-12,243.20
2025-09-15,246.84
2025-09-16,245.10
2025-09-17,247.42
2025-09-18,249.90
"""
ticker = "AAPL"

rows = csv_text.strip().splitlines()[1:]
closes = list(map(lambda row: float(row.split(",")[1]), rows))
rets = list(map(lambda pair: pair[1] / pair[0] - 1, zip(closes, closes[1:])))

mean_ret = sum(rets) / len(rets)
min_ret = min(rets)
max_ret = max(rets)


def as_pct(value):
    return f"{value * 100:+.2f}%"


mean_str = as_pct(mean_ret)
min_str = as_pct(min_ret)
max_str = as_pct(max_ret)

perf_line = f"{ticker},{len(rets)},{mean_str},{min_str},{max_str}"

first_date = rows[0].split(",")[0]
last_date = rows[-1].split(",")[0]
report_text = (
    f"{ticker} – Daily Close Snapshot\n"
    f"Dates: {first_date} → {last_date}\n"
    f"Observations: {len(rows)} (returns: {len(rets)}) | Mean/Min/Max: {mean_str} / {min_str} / {max_str}"
)

print("rows =", rows)
print("closes =", closes)
print("rets =", rets)
print("mean_str =", mean_str)
print("min_str =", min_str)
print("max_str =", max_str)
print("perf_line =", perf_line)
print(report_text)
print()

rows = ['2025-09-12,243.20', '2025-09-15,246.84', '2025-09-16,245.10', '2025-09-17,247.42', '2025-09-18,249.90']
closes = [243.2, 246.84, 245.1, 247.42, 249.9]
rets = [0.01496710526315792, -0.007049100631988381, 0.009465524275805715, 0.010023441920620879]
mean_str = +0.69%
min_str = -0.70%
max_str = +1.50%
perf_line = AAPL,4,+0.69%,-0.70%,+1.50%
AAPL – Daily Close Snapshot
Dates: 2025-09-12 → 2025-09-18
Observations: 5 (returns: 4) | Mean/Min/Max: +0.69% / -0.70% / +1.50%



## Question 5

Produce a single string status: 

In [ ]:
headlines = [
    "AAPL gains as Wedbush boosts price-target; BUY reiterated",
    "msft slips after DOJ update; Neutral → HOLD; PT trimmed",
    "GOOG jumps on cloud deal wins–Upgrade to BUY!",
    "NVDA downgraded to HOLD; valuation 'stretched'? ",
    "brk.b: Berkshire raises cash pile–Sell off in cyclicals?",
    "Spy edges higher; flows strong; BUY-the-dip chatter",
]
signal_buy = {"BUY", "UPGRADE", "ACCUMULATE", "OVERWEIGHT"}
signal_sell = {"SELL", "DOWNGRADE", "UNDERWEIGHT", "REDUCE"}
signal_hold = {"HOLD", "NEUTRAL", "MARKET PERFORM"}


# Uppercase every headline and standardize common Unicode dash characters to ASCII hyphen
# Added a space after comma/semicolon/colon when immediately followed by a letter
def clean_one(line):
    line = line.upper().replace("–", "-").replace("—", "-")
    line = re.sub(r"\s*([,;:])\s*(?=[A-Z])", r"\1 ", line)
    # don't split price-target or buy-the-dip
    line = re.sub(r"(?<![A-Z0-9])-\s*(?=[A-Z])", "- ", line)
    return line


clean_lines = list(map(clean_one, headlines))

token_re = re.compile(r"\^?[A-Z0-9]+(?:[.\-][A-Z0-9]+)*")
tokens = sum(map(token_re.findall, clean_lines), [])

# count the words BUY/SELL/HOLD, not UPGRADE/DOWNGRADE from the signal sets
counts = Counter(re.findall(r"\bBUY\b|\bSELL\b|\bHOLD\b", " ".join(clean_lines)))
buy_hits = counts["BUY"]
sell_hits = counts["SELL"]
hold_hits = counts["HOLD"]

# Exact token membership prevents partial matches. set() removes duplicates and sorted() orders them
mentioned = sorted(set(tokens) & set(universe))

# Format the headline count, signal counts and comma joined ticker list exactly as specified
headline_summary = (
    f"{len(headlines)} headlines | BUY: {buy_hits}, HOLD: {hold_hits}, SELL: {sell_hits} "
    f"| Tickers: {','.join(mentioned)}"
)

print("clean_lines =", clean_lines)
print("tokens =", tokens)
print("buy_hits =", buy_hits, " sell_hits =", sell_hits, " hold_hits =", hold_hits)
print("mentioned =", mentioned)
print("headline_summary =", headline_summary)


clean_lines = ['AAPL GAINS AS WEDBUSH BOOSTS PRICE-TARGET; BUY REITERATED', 'MSFT SLIPS AFTER DOJ UPDATE; NEUTRAL → HOLD; PT TRIMMED', 'GOOG JUMPS ON CLOUD DEAL WINS-UPGRADE TO BUY!', "NVDA DOWNGRADED TO HOLD; VALUATION 'STRETCHED'? ", 'BRK.B: BERKSHIRE RAISES CASH PILE-SELL OFF IN CYCLICALS?', 'SPY EDGES HIGHER; FLOWS STRONG; BUY-THE-DIP CHATTER']
tokens = ['AAPL', 'GAINS', 'AS', 'WEDBUSH', 'BOOSTS', 'PRICE-TARGET', 'BUY', 'REITERATED', 'MSFT', 'SLIPS', 'AFTER', 'DOJ', 'UPDATE', 'NEUTRAL', 'HOLD', 'PT', 'TRIMMED', 'GOOG', 'JUMPS', 'ON', 'CLOUD', 'DEAL', 'WINS-UPGRADE', 'TO', 'BUY', 'NVDA', 'DOWNGRADED', 'TO', 'HOLD', 'VALUATION', 'STRETCHED', 'BRK.B', 'BERKSHIRE', 'RAISES', 'CASH', 'PILE-SELL', 'OFF', 'IN', 'CYCLICALS', 'SPY', 'EDGES', 'HIGHER', 'FLOWS', 'STRONG', 'BUY-THE-DIP', 'CHATTER']
buy_hits = 3  sell_hits = 1  hold_hits = 2
mentioned = ['AAPL', 'BRK.B', 'GOOG', 'MSFT', 'NVDA', 'SPY']
headline_summary = 6 headlines | BUY: 3, HOLD: 2, SELL: 1 | Tickers: AAPL,BRK.B,GOOG,MSFT,NVDA